# Phase 3: Article Recommendation Engine with Entity Visualisation

## What this notebook does

A user types any keyword a person's name, an organisation, a location.
The engine finds all matching BBC articles and for each result renders
the article text with ALL named entities highlighted inline using
spaCy's displacy visualiser.

## What the output looks like

For a search of "Tony Blair":
- Article header: filename | category → sub-category | score
- Full article text rendered with colour-coded entity labels:
  [Tony Blair] PERSON  [Labour] ORG  [Iraq] GPE  [2004] DATE

## Import & Environment

In [ ]:
# Imports

import pandas as pd
import spacy
from spacy import displacy
from IPython.display import display, HTML

## Load spaCy model and all five NER DataFrames

We reload the same spaCy pipeline used in Phase 2.
`displacy.render()` needs a live spaCy Doc object — it can't
render from a stored string. So for each article the search
returns, we re-run `nlp(text)` to produce the Doc, then
pass it to displacy.

In [ ]:
# Load the spaCy English pipeline

nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded:", nlp.meta["name"])

# Load all five NER DataFrames and combine into one



BASE = '../Data'

# Load each category's NER output
dfs = {
    'Business':      pd.read_csv(BASE + 'NER_Business_Dataframe.csv'),
    'Politics':         pd.read_csv(BASE + 'NER_Politics_Dataframe.csv'),
    'Entertainment': pd.read_csv(BASE + 'NER_Entertainment_Dataframe.csv'),
    'Tech':      pd.read_csv(BASE + 'NER_Tech_Dataframe.csv'),
    'Sport':          pd.read_csv(BASE + 'NER_Sport_Dataframe.csv'),
}

# Stack into one master DataFrame
master_df = pd.concat(dfs.values(), ignore_index=True)

# Fill any NaN values in entity columns with empty string
for col in ['people', 'people_roles', 'organisations', 'locations',
            'money_refs', 'dates']:
    if col in master_df.columns:
        master_df[col] = master_df[col].fillna('None')

print(f"Master DataFrame shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()}")
print(f"\nArticles per category:")
print(master_df['category'].value_counts())

## Build the search function

Searches across four entity columns with different weights.
Returns the top N matching articles sorted by relevance score.

The search is case-insensitive — "tony blair", "Tony Blair",
and "TONY BLAIR" all return the same results.

- `people` highest weight (3) — direct name match
- `organisations`  high weight (3) — direct org match
- `locations` medium weight (2) — location match
- `preview` low weight (1) — raw text mention

**Why different weights?**
A person's name appearing in the `people` column means spaCy explicitly
identified them as a named entity in that article. That is a stronger
signal than the keyword just happening to appear somewhere in the text.
Weighting entity columns higher surfaces the most relevant articles first.

The function also shows *which field* the match came from, so results
are transparent and explainable.

In [ ]:
# Search function
# Returns a ranked DataFrame of matching articles

def search_articles(keyword, df, top_n=5):
    """
    Search all BBC articles for a keyword across entity columns.

    Scoring weights:
      people column       → 3  (spaCy explicitly tagged this as a person)
      organisations column → 3  (spaCy explicitly tagged this as an org)
      locations column    → 2  (spaCy explicitly tagged this as a location)
      people_roles column → 1  (role label match)
      preview text        → 1  (raw text fallback)

    Returns top_n results sorted by score descending,
    then confidence_score descending for tied scores.
    """
    kw = keyword.strip().lower()
    if not kw:
        return pd.DataFrame()

    results = []

    for _, row in df.iterrows():
        score = 0
        matched_fields = []

        checks = [
            ('people',        3),
            ('organisations', 3),
            ('locations',     2),
            ('people_roles',  1),
            ('preview',       1),
        ]

        for col, weight in checks:
            val = str(row.get(col, '')).lower()
            if kw in val:
                score += weight
                matched_fields.append(col)

        if score > 0:
            results.append({
                'score':         score,
                'matched_in':    ', '.join(matched_fields),
                'filename':      row.get('filename', ''),
                'category':      row.get('category', ''),
                'sub_category':  row.get('sub_category', ''),
                'confidence':    row.get('confidence_score', 0.0),
                'people':        str(row.get('people', 'None')),
                'organisations': str(row.get('organisations', 'None')),
                'locations':     str(row.get('locations', 'None')),
                'preview':       str(row.get('preview', '')),
            })

    if not results:
        return pd.DataFrame()

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(
        ['score', 'confidence'],
        ascending=[False, False]
    ).reset_index(drop=True)

    return results_df.head(top_n)

## displacy render function

This is the core visual output.

For each matched article, we:
1. Re-run `nlp(text)` to produce a live spaCy Doc object
2. Call `displacy.render(doc, style="ent")` which returns an
   HTML string with every entity wrapped in a coloured span
3. Display a header block showing the article metadata
4. Display the rendered entity HTML inline in the notebook

The colour coding is spaCy's default scheme:
- PERSON → blue
- ORG    → pink/orange
- GPE    → green (countries, cities)
- DATE   → yellow
- MONEY  → teal

In [ ]:
def render_article_entities(row, rank):
    """
    Takes one result row and renders the full article text
    with spaCy displacy entity highlighting.

    Parameters
    ----------
    row : pd.Series
        One row from the search results DataFrame.
    rank : int
        The result rank number (1, 2, 3 ...) for the header.
    """

    # Article header
    # Displays metadata above the rendered text so the user knows
    # which article they're looking at before reading it
    header_html = f"""
    <div style="
        border-left: 4px solid #4a90d9;
        padding: 10px 16px;
        margin: 24px 0 8px 0;
        background: #f7f9fc;
        border-radius: 4px;
        font-family: monospace;
    ">
        <span style="font-size:13px; color:#888;">Result {rank}</span><br>
        <strong style="font-size:15px;">
            {row['filename']}
        </strong>
        &nbsp;&nbsp;|&nbsp;&nbsp;
        <span style="color:#4a90d9;">{row['category']}</span>
        &nbsp;→&nbsp;
        <span style="color:#2ecc71;">{row['sub_category']}</span>
        <br>
        <span style="font-size:12px; color:#555;">
            Relevance score: <strong>{row['score']}</strong>
            &nbsp;&nbsp;|&nbsp;&nbsp;
            Matched in: <em>{row['matched_in']}</em>
            &nbsp;&nbsp;|&nbsp;&nbsp;
            Confidence: {round(float(row['confidence']), 4) if row['confidence'] else 0.0}
        </span>
        <br>
        <span style="font-size:12px; color:#555;">
            People: {row['people'][:100]}
        </span>
        <br>
        <span style="font-size:12px; color:#555;">
            Orgs: {row['organisations'][:100]}
        </span>
    </div>
    """
    display(HTML(header_html))

    #  Run spaCy NER on the article text
    # We re-run nlp() here because displacy needs a live Doc object.
    # The stored CSV columns hold string representations,those
    # can't be passed to displacy directly. nlp(text) takes ~0.1s
    # per article so it's fast enough for interactive search.
    text = row['preview']
    doc  = nlp(text)

    # Render entities inline
    # style="ent" tells displacy to render named entity labels.
    # jupyter=True returns an HTML string instead of opening a browser.
    # We wrap it in display(HTML(...)) to render it in the notebook.
    entity_html = displacy.render(doc, style="ent", jupyter=False)

    # Wrap in a div with some padding for readability
    wrapped = f"""
    <div style="
        padding: 12px 16px;
        background: #fff;
        border: 1px solid #e0e0e0;
        border-radius: 4px;
        font-size: 14px;
        line-height: 2;
        font-family: Georgia, serif;
        color: #000;
    ">
        {entity_html}
    </div>
    """
    display(HTML(wrapped))

## Main recommend function

Ties everything together. The user calls `recommend("keyword")`
and gets:
- A summary header showing how many articles were found
- Each article rendered with full entity highlighting
- A compact summary table at the bottom for quick scanning

In [ ]:
# Main recommend function

def recommend(keyword, top_n=5):
    """
    Search BBC articles by keyword and render entity-highlighted results.

    Each result shows:
    - Article metadata header (filename, category, sub-category, score)
    - Full article text with ALL named entities highlighted and labelled
      using spaCy displacy (PERSON, ORG, GPE, DATE, MONEY etc.)
    - Summary table at the bottom

    Usage:
        recommend("Tony Blair")
        recommend("Chelsea")
        recommend("Microsoft", top_n=3)
        recommend("Iraq", top_n=10)
    """

    # Search header
    display(HTML(f"""
    <div style="
        background: #1a1a2e;
        color: white;
        padding: 16px 20px;
        border-radius: 6px;
        font-family: monospace;
        margin-bottom: 8px;
    ">
        <span style="font-size:11px; color:#aaa;">BBC NLP — Recommendation Engine</span><br>
        <span style="font-size:18px; font-weight:bold;">
            Search: "{keyword}"
        </span>
    </div>
    """))

    # Run search
    results = search_articles(keyword, master_df, top_n=top_n)

    if results.empty:
        display(HTML(f"""
        <div style="color:#e74c3c; padding:12px; font-family:monospace;">
            No articles found containing '{keyword}'.
        </div>
        """))
        return

    display(HTML(f"""
    <div style="
        font-family: monospace;
        color: #000000;
        font-size: 13px;
        margin-bottom: 16px;
    ">
        Found <strong>{len(results)}</strong> relevant articles
        (showing top {top_n})
    </div>
    """))

    # Render each result with displacy
    # Each article gets a header block and a full entity-highlighted
    # rendering of its text
    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        render_article_entities(row, rank)

    #  Summary table
    # Compact table at the bottom for quick scanning across results
    display(HTML("<h4 style='margin-top:32px; font-family:monospace;'>Summary Table</h4>"))
    display(results[[
        'score', 'matched_in', 'filename',
        'category', 'sub_category', 'confidence'
    ]])

## Run searches by calling the `recommend` function and pass in a keyword E.g 'chelsea', 'Will Smith'

In [ ]:
# Run searches

# Person search — displacy will highlight Tony Blair as PERSON,
# Labour as ORG, Iraq/UK as GPE throughout each article
recommend("Tony Blair")

In [ ]:
# Run searches

# Person search — displacy will highlight Tony Blair as PERSON,
# Labour as ORG, Iraq/UK as GPE throughout each article
recommend("Alex Ferguson")